# O código abaixo está na ordem necessária para a execução. 
### É usado o Ollama, então será necessário fazer a instalação.

In [64]:
from pathlib import Path
import sqlite3

import logging
import os

In [77]:
from scripts.parser_pdf import parserPdf
from scripts.sqlite import criarBanco
from scripts.embeddings import gerarEmbeddings
from scripts.text_to_sql import textToSQL
from scripts.router import router
from scripts.answer import answer
from scripts.cleanText import cleanText
from scripts.crossEncoder import crossEncoder

In [66]:
import logging

# Silencia logs
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.WARNING)

# PERGUNTA DO USUÁRIO

In [ ]:
#query = "Qual é o instrumento mais barato e o mais caro?"
#query = "Qual é o prazo de devolução?"
#query = "Qual é o prazo para envio via sedex?"
#query = "Quantos Estados tem no Brasil?"
query = "Quais opções de violões disponíveis custando até R$1000?"

# PARÂMETROS

In [78]:
pdf_dir = Path("../data/data_pdf")
csv_dir = Path("../data/data_csv")
chunks_dir = Path("../processed_data/chunks")
bd_dir = "../data/dados.db"
embeddings_dir = "../processed_data/embeddings.npy"

k_rank = 5
k_rerank = 1

modelo = "BAAI/bge-m3"
#modelo_llm = "qwen2.5-coder:7b"
modelo_llm = "Qwen3:8B"
modelo_rerank = "BAAI/bge-reranker-v2-m3"

temperature = 0.3

conexao = sqlite3.connect(bd_dir)

# CRIAÇÃO DO BANCO DE DADOS, CHUNK DO PDF e GERAÇÃO DOS EMBEDDINGS.
- Só é necessário executar está etapa uma única vez.

In [75]:
parserPdf(pdf_dir, chunks_dir)
criarBanco(csv_dir, conexao)
gerarEmbeddings(modelo, chunks_dir, embeddings_dir)

Encontrados 1 PDF(s).
Processando: politicas_da_loja.pdf
  → 8 chunks criados.


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 38923.71it/s]


Carregando: politicas_da_loja.jsonl


Batches: 100%|██████████| 1/1 [00:11<00:00, 11.43s/it]


# Roteamento de consulta (Query Routing)

In [84]:
route = router(query, modelo_llm, temperature)
print(route)

HYBRID


# RANQUEAMENTO
- De acordo com o router, está etapa irá gerar os contextos sendo eles somente de RAG, somente SQL ou ambos.

In [80]:
contextoRAG = ""
resultadoSQL = ""

match route:
    case "RAG":
        print("Executando RAG...")
        resultadoRAG = crossEncoder(chunks_dir, query, modelo, modelo_rerank, embeddings_dir, k_rank, k_rerank)
        contextoRAG = "\n\n".join(result["text"] for result in resultadoRAG)

    case "SQL":
        print("Executando Text-to-SQL...")
        sql_gerado, cols, registros_sql = textToSQL(query, modelo_llm, temperature, conexao)
        resultadoSQL = f"{cols}\n{registros_sql}"

    case _:
        print("Executando RAG e Text-To-SQL...")
        resultadoRAG = crossEncoder(chunks_dir, query, modelo, modelo_rerank, embeddings_dir, k_rank, k_rerank)
        contextoRAG = "\n\n".join(result["text"] for result in resultadoRAG)

        sql_gerado, cols, registros_sql = textToSQL(query, modelo_llm, temperature, conexao)
        resultadoSQL = f"{cols}\n{registros_sql}"

Executando Text-to-SQL...


# GERA A RESPOSTA FINAL

In [81]:
contextoRAG = cleanText(contextoRAG)
resposta_final = answer(contextoRAG, resultadoSQL, query, modelo_llm, temperature)

# Mostra as resposta
- Resultados do reranking (somente o rank, sem os textos em si)
- Resultados do SQL (sql gerado e os registros gerados)
- Resultado final que o usuário veria

In [82]:
match route:
    case "RAG":
        print("\n\n--- Resultados do reranking ---")
        for result in resultadoRAG:
            print(
                f"\nScore: {result['score']:.4f}"
            )
            print(
                f"Documento: {result['source']}"
            )
            print(
                f"Página: {result['page']}"
            )
            print(
                f"Chunk: {result['chunk_id']}"
            )

    case "SQL":
        print("\n\n--- Resultados do SQL ---")
        print(sql_gerado)
        print("\nColunas:", cols)
        for row in registros_sql:
            print(row)

    case _:
        print("\n\n--- Resultados do reranking ---")
        for result in resultadoRAG:
            print(
                f"\nScore: {result['score']:.4f}"
            )
            print(
                f"Documento: {result['source']}"
            )
            print(
                f"Página: {result['page']}"
            )
            print(
                f"Chunk: {result['chunk_id']}"
            )

        print("\n\n--- Resultados do SQL ---")
        print(sql_gerado)
        print("\nColunas:", cols)
        for row in registros_sql:
            print(row)

print(f"\n--- Resposta final ---\n\n{resposta_final}")



--- Resultados do SQL ---
SELECT p.product_id, p.name, p.description, p.price_brl, c.name AS category, p.specs
FROM products p
JOIN categories c ON p.category_id = c.category_id
WHERE p.status = 'active'
  AND (
      p.name LIKE '%violão%'
      OR p.name LIKE '%acústico%'
      OR p.description LIKE '%violão%'
      OR p.description LIKE '%acústico%'
      OR c.name LIKE '%violão%'
      OR c.name LIKE '%acústico%'
      OR c.description LIKE '%violão%'
      OR c.description LIKE '%acústico%'
  )
  AND p.price_brl <= 1000;

Colunas: ['product_id', 'name', 'description', 'price_brl', 'category', 'specs']
(81, 'Yamaha C40 Nylon Natural', 'Violão clássico com tampo em Spruce e fundo/laterais em Meranti. Ideal para estudantes e iniciantes, com excelente projeção sonora.', 599.9, 'Violões', '{"top":"Spruce","back_sides":"Meranti","neck":"Nato","strings":"nylon","scale":"650mm","electronics":"no","color":"Natural"}')
(82, 'Yamaha C70 Nylon Natural', 'Violão clássico de nível intermediár